<a href="https://colab.research.google.com/github/esero88/eser-portfolio/blob/main/pension-retention-analytics/notebooks/01_egm_veri_toplama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EGM Veri Toplama — BES Kalıcılık ve Rekabet Projesi

Emeklilik Gözetim Merkezi'nin (egm.org.tr) yayınladığı BES istatistik raporlarını
tarih tarih indirir. Her rapor tek bir tarihin anlık görüntüsüdür; zaman serisi
için tarih listesindeki her tarih tek tek geziliyor.

**Hedef raporlar**

| Kod | Rapor | Ne için |
|---|---|---|
| `GR-01-015` | Sözleşmelerin Yürürlüğe Giriş Ayına Göre Sonlanma Oranı | Kohort kalıcılık eğrisi (2003.10 →) |
| `GR-01-010` | Sonlanan Sözleşmelerin Sonlanma Nedenine Göre Dağılımı | Ayrılma nedenleri kırılımı |
| `GR-01-019` | Cayma Hakkı Kullanım Oranı | Cayma trendi |
| `EGM-05-006` | Emeklilik Şirketi ve Faiz Detayında Fon Büyüklükleri | Şirket bazında pazar payı |
| `EGM-05-001` | Fon Bazında Fon İşletim Gideri Kesinti Oranları | Maliyet rekabeti |

**Akış:** Kurulum → Script → Keşif (`--probe`) → Küçük deneme → Tam toplama → İndir

---

> **Kaynak:** Bu defterde toplanan tüm istatistiksel veriler Emeklilik Gözetim
> Merkezi A.Ş.'ye (EGM) aittir. Kaynak gösterilmeden alıntı yapılamaz.

## 1. Kurulum

Playwright ve Chromium kuruluyor. EGM sayfası JavaScript ile render edildiği için
`requests` boş tablo döndürür; gerçek bir tarayıcı motoru gerekiyor.

Ardından Google Drive bağlanıyor. İnen dosyalar doğrudan Drive'a yazılıyor —
Colab'ın geçici diski oturum kapanınca siliniyor, Drive'daki kalıyor. Toplama
yarıda kesilirse tekrar çalıştırmak yeterli; inmiş dosyalar atlanıyor.

Colab oturumu sıfırlandığında bu iki hücre tekrar çalıştırılmalı.

In [1]:
!pip install playwright -q
!playwright install chromium
!playwright install-deps chromium

191.1 MiB [] 0% 0.0s191.1 MiB [] 0% 23.8s191.1 MiB [] 0% 10.7s191.1 MiB [] 0% 4.8s191.1 MiB [] 1% 4.5s191.1 MiB [] 2% 2.9s191.1 MiB [] 3% 3.0s191.1 MiB [] 4% 2.6s191.1 MiB [] 4% 2.9s191.1 MiB [] 4% 3.1s191.1 MiB [] 4% 3.2s191.1 MiB [] 6% 2.6s191.1 MiB [] 7% 2.4s191.1 MiB [] 9% 2.2s191.1 MiB [] 10% 2.0s191.1 MiB [] 11% 1.9s191.1 MiB [] 12% 1.9s191.1 MiB [] 13% 1.8s191.1 MiB [] 13% 1.9s191.1 MiB [] 13% 2.0s191.1 MiB [] 13% 2.1s191.1 MiB [] 13% 2.2s191.1 MiB [] 15% 2.0s191.1 MiB [] 16% 1.9s191.1 MiB [] 18% 1.9s191.1 MiB [] 19% 1.8s191.1 MiB [] 20% 1.7s191.1 MiB [] 21% 1.7s191.1 MiB [] 21% 1.8s191.1 MiB [] 23% 1.7s191.1 MiB [] 25% 1.6s191.1 MiB [] 26% 1.5s191.1 MiB [] 28% 1.4s191.1 MiB [] 29% 1.3s191.1 MiB [] 31% 1.3s191.1 MiB [] 33% 1.2s191.1 MiB [] 35% 1.1s191.1 MiB [] 36% 1.1s191.1 MiB [] 38% 1.1s191.1 MiB [] 39% 1.1s191.1 MiB [] 40% 1.1s191.1 MiB [] 41% 1.1s191.1 MiB [] 43% 1.0s191.1 MiB [] 44% 1.0s191.1 MiB [] 46% 0.9s191.1 MiB [] 48% 0.9s191.1 MiB [] 49% 0.9s191.1 MiB [] 51% 0.8s191.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

HEDEF = "/content/drive/MyDrive/egm-bes-projesi/data/raw"
!mkdir -p "$HEDEF"
print("Hedef klasör:", HEDEF)

Mounted at /content/drive
Hedef klasör: /content/drive/MyDrive/egm-bes-projesi/data/raw


## 2. Toplayıcı script'i

Script repo'da (`scripts/egm_collector.py`) duruyor; buraya kopyalanmak yerine
oradan çekiliyor. Tek doğru nüsha repo olduğu için defterdeki kopya eskimiyor.

Ayrı bir dosya olarak çalıştırılmasının nedeni: Playwright'ın sync API'si
notebook'un asyncio döngüsüyle çakışıyor. `!python` ile ayrı süreç olarak
çalıştırıldığında sorun kalmıyor.

In [3]:
!wget -q -O egm_collector.py https://raw.githubusercontent.com/esero88/eser-portfolio/main/pension-retention-analytics/scripts/egm_collector.py
!wc -l egm_collector.py

368 egm_collector.py


## 3. Keşif

`--probe` hiçbir şey indirmez; sayfayı açıp iki şeyi raporlar:

- Tarih listesi bulundu mu, kaç tarih var
- "Excel'e Aktar" düğmesi görülebiliyor mu

`RAPOR_URL` değişkenine raporun EGM'deki adresi yazılıyor.

In [4]:
RAPOR_URL = "https://egm.org.tr/bilgi-merkezi/istatistikler?r=b5eea773-1617-4344-8e66-1771226ee845"   # <-- raporun linkini buraya yapıştır
RAPOR_KODU = "GR-01-015"

assert RAPOR_URL, "Önce RAPOR_URL'i doldur"
!python egm_collector.py --url "$RAPOR_URL" --code "$RAPOR_KODU" --probe

Aciliyor: https://egm.org.tr/bilgi-merkezi/istatistikler?r=b5eea773-1617-4344-8e66-1771226ee845
83 tarih bulundu.  Ilk: '29 Eylül 2026 Salı'  Son: '30 Aralık 2018 Pazar'
Excel dugmesi: BULUNDU

--- ilk 10 tarih ---
   29 Eylül 2026 Salı -> 2026-09-29
   30 Ağustos 2026 Pazar -> 2026-08-30
   30 Temmuz 2026 Perşembe -> 2026-07-30
   29 Haziran 2026 Pazartesi -> 2026-06-29
   30 Mayıs 2026 Cumartesi -> 2026-05-30
   29 Nisan 2026 Çarşamba -> 2026-04-29
   30 Mart 2026 Pazartesi -> 2026-03-30
   27 Şubat 2026 Cuma -> 2026-02-27
   30 Ocak 2026 Cuma -> 2026-01-30
   30 Aralık 2025 Salı -> 2025-12-30

Kesif modu, indirme yapilmadi.


### Çıktının okunması

| Çıktı | Anlamı |
|---|---|
| "N tarih bulundu" + "Excel düğmesi: BULUNDU" | Yolunda, toplamaya geçilebilir |
| "tarih listesi bulunamadı" | Tarih seçici normal bir `<select>` değil; `tarih_secicisini_bul()` güncellenmeli |
| "Excel düğmesi: BULUNAMADI" | `export_dugmesini_bul()` güncellenmeli |
| Zaman aşımı / bağlantı hatası | EGM, Colab'ın IP'sini engelliyor olabilir; script yerel makinede çalıştırılmalı |

## 4. Küçük deneme

Tam toplamadan önce üç tarihle sınanıyor: dosyalar iniyor mu, içerikleri dolu mu.

In [5]:
!python egm_collector.py --url "$RAPOR_URL" --code "$RAPOR_KODU" --out "$HEDEF" --limit 3

Aciliyor: https://egm.org.tr/bilgi-merkezi/istatistikler?r=b5eea773-1617-4344-8e66-1771226ee845
83 tarih bulundu.  Ilk: '29 Eylül 2026 Salı'  Son: '30 Aralık 2018 Pazar'
Excel dugmesi: BULUNDU
[1/3] 29 Eylül 2026 Salı ... -> GR-01-015_2026-09-29.xlsx
[2/3] 30 Ağustos 2026 Pazar ... -> GR-01-015_2026-08-30.xlsx
[3/3] 30 Temmuz 2026 Perşembe ... -> GR-01-015_2026-07-30.xlsx

Bitti.  indirilen: 3   atlanan: 0   hatali: 0
Klasor: /content/drive/MyDrive/egm-bes-projesi/data/raw/GR-01-015


In [6]:
import pandas as pd
from pathlib import Path

dosyalar = sorted(Path(HEDEF, RAPOR_KODU).glob("*.xlsx"))
print(f"{len(dosyalar)} dosya indi:")
for d in dosyalar[:5]:
    print(" ", d.name, f"({d.stat().st_size:,} byte)")
if len(dosyalar) > 5:
    print(f"  ... ve {len(dosyalar) - 5} dosya daha")

if dosyalar:
    ornek = pd.read_excel(dosyalar[-1], dtype=str)
    print(f"\n{dosyalar[-1].name}  ->  {ornek.shape[0]} satır × {ornek.shape[1]} kolon")
    display(ornek.head(10))

3 dosya indi:
  GR-01-015_2026-07-30.xlsx (14,767 byte)
  GR-01-015_2026-08-30.xlsx (12,509 byte)
  GR-01-015_2026-09-29.xlsx (12,542 byte)

GR-01-015_2026-09-29.xlsx  ->  276 satır × 3 kolon


,Unnamed: 0,Sonlanma Oranı (%),Yürürlüğe Giren Sözleşme Adedi
0,2003.10,55.41,388
1,2003.11,60.34,4715
2,2003.12,60.62,16983
3,2004.01,60.22,16284
4,2004.02,61.23,15917
5,2004.03,68.59,33662
6,2004.04,61.9,33077
7,2004.05,67.29,25660
8,2004.06,65.93,29553
9,2004.07,66.27,27635


## 5. Tam toplama

Tarih listesinin tamamı indiriliyor. Zaten inmiş dosyalar atlanıyor, yarıda
kesilirse tekrar çalıştırıldığında kaldığı yerden devam ediyor.

`--delay 2.5` istekler arasındaki bekleme; siteyi yormamak için bırakılmıştır.

In [7]:
!python egm_collector.py --url "$RAPOR_URL" --code "$RAPOR_KODU" --out "$HEDEF" --delay 2.5

Aciliyor: https://egm.org.tr/bilgi-merkezi/istatistikler?r=b5eea773-1617-4344-8e66-1771226ee845
83 tarih bulundu.  Ilk: '29 Eylül 2026 Salı'  Son: '30 Aralık 2018 Pazar'
Excel dugmesi: BULUNDU
[4/83] 29 Haziran 2026 Pazartesi ... -> GR-01-015_2026-06-29.xlsx
[5/83] 30 Mayıs 2026 Cumartesi ... -> GR-01-015_2026-05-30.xlsx
[6/83] 29 Nisan 2026 Çarşamba ... -> GR-01-015_2026-04-29.xlsx
[7/83] 30 Mart 2026 Pazartesi ... -> GR-01-015_2026-03-30.xlsx
[8/83] 27 Şubat 2026 Cuma ... -> GR-01-015_2026-02-27.xlsx
[9/83] 30 Ocak 2026 Cuma ... -> GR-01-015_2026-01-30.xlsx
[10/83] 30 Aralık 2025 Salı ... -> GR-01-015_2025-12-30.xlsx
[11/83] 29 Kasım 2025 Cumartesi ... -> GR-01-015_2025-11-29.xlsx
[12/83] 30 Ekim 2025 Perşembe ... -> GR-01-015_2025-10-30.xlsx
[13/83] 29 Eylül 2025 Pazartesi ... -> GR-01-015_2025-09-29.xlsx
[14/83] 30 Ağustos 2025 Cumartesi ... -> GR-01-015_2025-08-30.xlsx
[15/83] 30 Temmuz 2025 Çarşamba ... -> GR-01-015_2025-07-30.xlsx
[16/83] 29 Haziran 2025 Pazar ... -> GR-01-015_2

## 6. Diğer raporlar

Her raporun kendi adresi var. Aynı işlem, rapor kodu ve link değiştirilerek
tekrarlanıyor.

In [8]:
# Her rapor için: linki zincir ikonundan ("Rapor adresini kopyala") al,
# kodu sayfadaki gri yazıdan teyit et.
RAPOR_URL_2  = "https://egm.org.tr/bilgi-merkezi/istatistikler?r=f6180d36-fa4f-426d-9909-1ebe29deeab4"             # <-- raporun linki
RAPOR_KODU_2 = "EGM-05-006"   # <-- raporun kodu

if RAPOR_URL_2:
    !python egm_collector.py --url "$RAPOR_URL_2" --code "$RAPOR_KODU_2" --out "$HEDEF" --delay 2.5
else:
    print("RAPOR_URL_2 boş — doldurulunca çalışır.")

Aciliyor: https://egm.org.tr/bilgi-merkezi/istatistikler?r=f6180d36-fa4f-426d-9909-1ebe29deeab4
72 tarih bulundu.  Ilk: '29 Eylül 2026 Salı'  Son: '30 Temmuz 1925 Perşembe'
Excel dugmesi: BULUNDU
[1/72] 29 Eylül 2026 Salı ... -> EGM-05-006_2026-09-29.xlsx
[2/72] 30 Ağustos 2026 Pazar ... -> EGM-05-006_2026-08-30.xlsx
[3/72] 30 Temmuz 2026 Perşembe ... -> EGM-05-006_2026-07-30.xlsx
[4/72] 29 Haziran 2026 Pazartesi ... -> EGM-05-006_2026-06-29.xlsx
[5/72] 30 Mayıs 2026 Cumartesi ... -> EGM-05-006_2026-05-30.xlsx
[6/72] 29 Nisan 2026 Çarşamba ... -> EGM-05-006_2026-04-29.xlsx
[7/72] 30 Mart 2026 Pazartesi ... -> EGM-05-006_2026-03-30.xlsx
[8/72] 27 Şubat 2026 Cuma ... -> EGM-05-006_2026-02-27.xlsx
[9/72] 30 Ocak 2026 Cuma ... -> EGM-05-006_2026-01-30.xlsx
[10/72] 30 Aralık 2025 Salı ... -> EGM-05-006_2025-12-30.xlsx
[11/72] 29 Kasım 2025 Cumartesi ... -> EGM-05-006_2025-11-29.xlsx
[12/72] 30 Ekim 2025 Perşembe ... -> EGM-05-006_2025-10-30.xlsx
[13/72] 29 Eylül 2025 Pazartesi ... -> EGM-05

## 7. Kontrol

Dosyalar Drive'da; indirmeye gerek yok. Bu hücre hangi rapordan kaç dosya
toplandığını gösterir.

In [ ]:
from pathlib import Path

kok = Path(HEDEF)
if not kok.exists():
    print("Henüz dosya yok.")
else:
    toplam_dosya = 0
    for klasor in sorted(k for k in kok.iterdir() if k.is_dir()):
        dosyalar = list(klasor.glob("*.xlsx"))
        boyut = sum(d.stat().st_size for d in dosyalar) / 1024
        toplam_dosya += len(dosyalar)
        print(f"{klasor.name:<14} {len(dosyalar):>4} dosya   {boyut:>8,.0f} KB")
    print(f"\nToplam: {toplam_dosya} dosya")

---

## Sonraki adım

Ham dosyalar toplandıktan sonra:

1. **Temizlik** — iki satıra bölünmüş başlıkların düzleştirilmesi, bilimsel
   gösterimdeki sayıların düzeltilmesi, uzun formata (`rapor_tarihi, şirket,
   segment, metrik, değer`) çevrilmesi.
2. **Ambar** — temiz tabloların PostgreSQL'e yüklenmesi, yıldız şema.
3. **Dashboard** — Tableau: kohort kalıcılık eğrisi, sonlanma nedenleri,
   pazar payı, maliyet karşılaştırması.
4. **Agent** — metrik sözlüğü, doğal dilden SQL, 30 soruluk doğruluk testi.